# 🧪 Sesión 23 · Operaciones CRUD Transaccionales y Async Sessions

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué esperar a la base de datos no debe bloquear el servidor?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros  {"titulo":"Rayuela","anio":1963}  -> 201
GET  /api/libros                                  -> 200
GET  /api/libros/1                                -> 200
PUT  /api/libros/1  {"titulo":"Rayuela (edición)"} -> 200
DELETE /api/libros/1                              -> 204
GET  /api/libros/1                                -> 404
```

_La base de datos sigue siendo SQLite; lo que cambia es cómo la aplicación espera sus respuestas._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un CRUD con SQLAlchemy asíncrono.
- Usar AsyncSession y aiosqlite.
- Verificar 201, 200, 404 y 204 en el mismo flujo.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Crear un libro → HTTP 201
- [ ] Listar los libros → HTTP 200
- [ ] Actualizar el libro → HTTP 200
- [ ] Eliminar el libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] La persistencia es asíncrona

## 🧩 El reto

Implementa un CRUD completo con SQLAlchemy asíncrono y aiosqlite. El servidor no debe bloquearse mientras espera la base.

**Requisitos del contrato**

- Usa create_async_engine con sqlite+aiosqlite.
- Cada endpoint usa AsyncSession y await.
- POST responde 201; GET 200; PUT 200; DELETE 204.
- Un id inexistente responde 404.
- La tabla se crea al iniciar la aplicación.

> **Pista conceptual:** Si tu endpoint es `def` y usas una sesión síncrona, estás bloqueando el event loop. La regla es: motor async, sesión async, endpoint async.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion23.md` con la diferencia entre def y async def.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
"""Sesión 23 · CRUD asíncrono: SQLAlchemy 2.0 + AsyncSession + aiosqlite."""
from contextlib import asynccontextmanager
from typing import Annotated, AsyncIterator

from fastapi import Depends, FastAPI, HTTPException, Response, status
from pydantic import BaseModel, ConfigDict, Field
from sqlalchemy import String, select
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker, create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# El driver aiosqlite permite que esperar a SQLite no bloquee el event loop
engine = create_async_engine("sqlite+aiosqlite:///biblioteca.db")
SesionLocal = async_sessionmaker(engine, expire_on_commit=False)   # objetos usables tras commit


class Base(DeclarativeBase):
    pass


class Libro(Base):
    __tablename__ = "libros"
    id: Mapped[int] = mapped_column(primary_key=True)
    titulo: Mapped[str] = mapped_column(String(200))
    anio: Mapped[int]


_esquema_listo = False


async def crear_esquema() -> None:
    """create_all es síncrono: run_sync lo ejecuta dentro de la conexión asíncrona."""
    global _esquema_listo
    if not _esquema_listo:
        async with engine.begin() as conn:
            await conn.run_sync(Base.metadata.create_all)
        _esquema_listo = True


@asynccontextmanager
async def lifespan(app: FastAPI):
    await crear_esquema()
    yield
    await engine.dispose()           # cierra el pool al apagar


app = FastAPI(title="CRUD asíncrono · DAB-2201", version="1.0.0", lifespan=lifespan)


async def get_db() -> AsyncIterator[AsyncSession]:
    """Una sesión por petición; se cierra siempre (Unit of Work)."""
    await crear_esquema()            # también funciona con TestClient sin lifespan
    async with SesionLocal() as sesion:
        yield sesion


DB = Annotated[AsyncSession, Depends(get_db)]


class LibroEntrada(BaseModel):
    model_config = ConfigDict(extra="forbid")
    titulo: str = Field(..., min_length=1, max_length=200)
    anio: int = Field(..., ge=1450, le=2100)


class LibroSalida(LibroEntrada):
    model_config = ConfigDict(from_attributes=True)
    id: int


async def buscar(db: AsyncSession, libro_id: int) -> Libro:
    libro = await db.get(Libro, libro_id)
    if libro is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "libro no encontrado")
    return libro


@app.post("/api/libros", status_code=status.HTTP_201_CREATED, response_model=LibroSalida)
async def crear(datos: LibroEntrada, db: DB):
    libro = Libro(**datos.model_dump())
    db.add(libro)                    # Pending
    await db.commit()                # Persistent: se escribe en la base
    await db.refresh(libro)          # trae el id generado
    return libro


@app.get("/api/libros")
async def listar(db: DB) -> dict:
    resultado = await db.scalars(select(Libro).order_by(Libro.id))
    return {"libros": [LibroSalida.model_validate(l).model_dump() for l in resultado.all()]}


@app.get("/api/libros/{libro_id}", response_model=LibroSalida)
async def obtener(libro_id: int, db: DB):
    return await buscar(db, libro_id)


@app.put("/api/libros/{libro_id}", response_model=LibroSalida)
async def actualizar(libro_id: int, datos: LibroEntrada, db: DB):
    libro = await buscar(db, libro_id)
    for campo, valor in datos.model_dump().items():
        setattr(libro, campo, valor)  # el ORM solo emite UPDATE de lo que cambió
    await db.commit()
    return libro


@app.delete("/api/libros/{libro_id}", status_code=status.HTTP_204_NO_CONTENT)
async def eliminar(libro_id: int, db: DB):
    libro = await buscar(db, libro_id)
    await db.delete(libro)
    await db.commit()
    return Response(status_code=status.HTTP_204_NO_CONTENT)


Writing app.py


## 🧪 Tu contrato C-R-E-O

| Elemento | Mi definición |
| :--- | :--- |
| **Contexto** | FastAPI + SQLAlchemy 2.0 asíncrono (`create_async_engine`, `async_sessionmaker`, `AsyncSession`) con el driver `aiosqlite` sobre `biblioteca.db`. |
| **Rol** | Desarrollador backend que migra un CRUD síncrono a uno asíncrono sin bloquear el event loop. |
| **Especificación** | `POST /api/libros` → 201 con `id`; `GET /api/libros` → 200 `{"libros": [...]}`; `GET /api/libros/{id}` → 200 o 404; `PUT /api/libros/{id}` → 200; `DELETE /api/libros/{id}` → 204 sin cuerpo; una sesión por petición con `Depends` y `await` en `execute`, `commit` y `refresh`. |
| **Output** | Solo `app.py`. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Crear un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Listar los libros',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'Actualizar el libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Rayuela (edición crítica)', 'anio': 1963},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela (edición crítica)'}},
 {'descripcion': 'Eliminar el libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. Crear un libro
✅ 02. Listar los libros
✅ 03. Actualizar el libro
✅ 04. Eliminar el libro
✅ 05. El libro eliminado ya no existe
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["AsyncSession|async_sessionmaker|aiosqlite"], descripcion="La persistencia es asíncrona"))
resumen(resultados)


✅ La persistencia es asíncrona
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion23.md` a `auditorias/auditoria_sesion23.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
